# Atención

Atención multi-cabeza, atención suave con RL, atención local y atención a la salida.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

> **Medido en el banco** (`laboratorio/banco/`): ver la sección «cuaderno 05» de `laboratorio/banco/README.md`. La atención suave con RL se mide dentro de un Transformer (`transformer_mascara_rl`) contra el Transformer normal.

## Atención multi-cabeza (la estándar, como referencia)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# Paso 1: Implementación de una sola cabeza de atención
class AttentionHead(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        d_k = Q.shape[-1]
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_weights, V)

        return output, attn_weights

# Paso 2: Multi-head attention desde cero
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, E = x.shape
        H = self.num_heads
        D = self.head_dim

        Q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        K = self.k_proj(x).view(B, T, H, D).transpose(1, 2)
        V = self.v_proj(x).view(B, T, H, D).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (D ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        out = torch.matmul(attn_weights, V)

        out = out.transpose(1, 2).contiguous().view(B, T, E)
        out = self.out_proj(out)

        return out, attn_weights  # (B, H, T, T)

# Paso 3: Visualización de mapas de atención
def plot_attention_weights(attn_weights, tokens=None):
    heads = attn_weights.shape[1]
    fig, axes = plt.subplots(1, heads, figsize=(16, 4))
    tokens = tokens or [f"tok_{i}" for i in range(attn_weights.shape[-1])]

    for i in range(heads):
        sns.heatmap(attn_weights[0, i].detach().numpy(),
                    xticklabels=tokens,
                    yticklabels=tokens,
                    cmap="Blues",
                    ax=axes[i])
        axes[i].set_title(f'Head {i+1}')
        axes[i].set_xlabel("Keys")
        axes[i].set_ylabel("Queries")

    plt.tight_layout()
    plt.show()

# Paso 4: Evaluación de diversidad entre cabezas
def similarity_between_heads(attn_weights):
    H = attn_weights.shape[1]
    similarities = torch.zeros(H, H)

    for i in range(H):
        for j in range(H):
            a = attn_weights[0, i].flatten()
            b = attn_weights[0, j].flatten()
            similarities[i, j] = F.cosine_similarity(a, b, dim=0)

    return similarities

# Simulación
torch.manual_seed(0)
seq_len = 5
embed_dim = 64
num_heads = 4

x = torch.randn(1, seq_len, embed_dim)
mha = MultiHeadAttention(embed_dim, num_heads)

# Simulamos entrenamiento guardando attn inicial y final
attn_first = None
attn_last = None
for epoch in range(10):
    out, attn_weights = mha(x)
    if epoch == 0:
        attn_first = attn_weights.clone()
    if epoch == 9:
        attn_last = attn_weights.clone()

attn_first.shape, attn_last.shape

## Atención suave con RL (texto)

Cada cabeza aprende una máscara de posiciones (qué token mira a cuál) con ruido Gumbel. Pese al nombre, la máscara aprende con el gradiente normal. Fallo de diseño: la máscara **multiplica** los puntajes, así que un valor ≈ 0 no bloquea la atención; en el banco hay una versión arreglada que suma log(máscara).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import random # Para simular datos

# Clase del modelo de atención (igual que la RevisedMultiRLSoftAttention que ya tienes)
class RevisedMultiRLSoftAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, seq_len, temperature=0.7):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.seq_len = seq_len
        self.temperature = temperature

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim) # Proyección de salida principal

        # RL masks - one set of parameters per head
        self.rl_masks = nn.ParameterList([nn.Parameter(torch.randn(seq_len, seq_len)) for _ in range(num_heads)])

        # --- Novedades para la nueva tarea: Capas de salida específicas ---
        # Clasificadores a nivel de token (ej. para Verbo/Sujeto/Predicado como etiquetas binarias por token)
        # Asumimos una clasificación de 3 tipos de elementos: Sujeto, Verbo, Predicado + 'Otro'
        self.token_classifier = nn.Linear(embed_dim, 4) # Output: logits para 4 clases (Sujeto, Verbo, Predicado, Otro)
        # Clasificador de sentimiento (ej. Positivo, Negativo, Neutro)
        self.sentiment_classifier = nn.Linear(embed_dim, 3) # Output: logits para 3 clases de sentimiento
        # Clasificador de directiva compleja - AGREGADO
        self.directive_classifier = nn.Linear(embed_dim, 1) # Output: logits para 1 clase (0=No, 1=Si)


    def forward(self, x, training=True):
        B, T, E = x.shape
        H = self.num_heads
        D = self.head_dim
        assert E == H * D, f"Input embedding dimension mismatch. Expected {H * D}, got {E}"
        assert T == self.seq_len, f"Input sequence length mismatch. Expected {self.seq_len}, got {T}"

        Q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        K = self.k_proj(x).view(B, T, H, D).transpose(1, 2)
        V = self.v_proj(x).view(B, T, H, D).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (D ** 0.5)

        masked_scores_list = []
        attn_masks_output_list = [] # This will store the softmaxed masks

        for i in range(H):
            rl_logits = self.rl_masks[i] # Shape (seq_len, seq_len)

            if training:
                # Generate Gumbel noise and apply Gumbel-Softmax trick
                # Noise should have the same shape as rl_logits
                gumbel_noise = -torch.log(-torch.log(torch.rand_like(rl_logits, device=rl_logits.device) + 1e-10) + 1e-10) # Add small epsilon for numerical stability
                mask_logits = (rl_logits + gumbel_noise) / self.temperature
            else:
                # In evaluation/inference, just use the deterministic mask
                mask_logits = rl_logits

            # Apply softmax along the dimension you want the mask probabilities to sum up to 1 (typically the last dimension)
            # This mask is applied element-wise to the scores matrix (TxT) for each head.
            attn_mask_per_head = F.softmax(mask_logits, dim=-1) # Shape (seq_len, seq_len), rows sum to 1

            # Store the resulting mask tensor for visualization/analysis
            attn_masks_output_list.append(attn_mask_per_head)

            # Expand mask to match batch and head dimensions for element-wise multiplication with scores
            # scores[:, i, :, :] has shape (B, T, T)
            # attn_mask_per_head has shape (T, T)
            # Need to expand attn_mask_per_head to (B, T, T) to multiply with scores[:, i, :, :]
            attn_mask_exp = attn_mask_per_head.unsqueeze(0).expand(B, -1, -1) # Shape (B, T, T)

            # Apply mask to scores for this head
            # masked_scores_per_head will have shape (B, T, T)
            masked_scores_per_head = scores[:, i, :, :] * attn_mask_exp # Element-wise multiplication

            # Append the masked scores for this head, adding back the head dimension
            masked_scores_list.append(masked_scores_per_head.unsqueeze(1)) # Shape (B, 1, T, T)


        # Concatenate masked scores along the head dimension
        masked_scores = torch.cat(masked_scores_list, dim=1) # Shape (B, H, T, T)

        # Apply softmax to the masked scores to get final attention weights
        # These attention weights sum to 1 along the last dimension for each query token and each head.
        attn_weights = F.softmax(masked_scores, dim=-1) # Shape (B, H, T, T)

        # Apply attention weights to V
        # (B, H, T, T) @ (B, H, T, D) -> (B, H, T, D)
        out = torch.matmul(attn_weights, V) # Shape (B, H, T, D)

        # Concatenate head outputs along the embedding dimension and apply final projection
        # (B, H, T, D) -> transpose (B, T, H, D) -> view (B, T, H*D) -> (B, T, E)
        out = out.transpose(1, 2).contiguous().view(B, T, E) # Shape (B, T, E)
        output_main = self.out_proj(out) # Salida principal del MHA (B, T, E)

        # --- Salidas específicas para la tarea ---
        # Predicciones a nivel de token para Sujeto/Verbo/Predicado/Otro
        token_logits = self.token_classifier(output_main) # Shape (B, T, 4)

        # Predicción de sentimiento (usamos pooling sobre la salida principal para obtener un vector de sentencia)
        sentence_embedding = torch.mean(output_main, dim=1) # Pooling simple (media), Shape (B, E)
        sentiment_logits = self.sentiment_classifier(sentence_embedding) # Shape (B, 3)

        # Predicción de directiva compleja - AGREGADO
        # Usamos el embedding de la sentencia para clasificar la directiva.
        directive_logits = self.directive_classifier(sentence_embedding) # Shape (B, 1)


        # Return the main output, the softmaxed RL masks, and the task-specific logits
        # attn_masks_output_list contains H tensors, each of shape (seq_len, seq_len)
        return output_main, attn_masks_output_list, token_logits, sentiment_logits, directive_logits


# --- Generación de Datos Dummy para la Nueva Tarea ---
# Esto es CRÍTICO: Necesitas datos que simulen oraciones y sus etiquetas
# Esto es solo un ejemplo MUY simplificado. En la realidad, usarías un dataset de PLN.
def generate_dummy_sentence_data(batch_size, seq_len, embed_dim):
    # Simula embeddings de palabras
    x_data = torch.randn(batch_size, seq_len, embed_dim)

    # Simula etiquetas de "ground truth" para la tarea (a nivel de token y oración)
    # Sujeto=0, Verbo=1, Predicado=2, Otro=3
    # Genera una secuencia de etiquetas aleatorias para cada token
    # Cada oración tendrá al menos un Sujeto (0), Verbo (1) y Predicado (2)
    y_token_gt = torch.full((batch_size, seq_len), 3, dtype=torch.long) # Inicialmente todo 'Otro'
    for b in range(batch_size):
        # Asigna un sujeto, verbo y predicado en posiciones aleatorias
        indices = random.sample(range(seq_len), min(3, seq_len)) # Select up to 3 unique indices
        if len(indices) > 0: y_token_gt[b, indices[0]] = 0 # Sujeto
        if len(indices) > 1: y_token_gt[b, indices[1]] = 1 # Verbo
        if len(indices) > 2: y_token_gt[b, indices[2]] = 2 # Predicado

    # Simula etiquetas de sentimiento (0=Positivo, 1=Negativo, 2=Neutro)
    y_sentiment_gt = torch.randint(0, 3, (batch_size,), dtype=torch.long)

    # Simula si hay una directiva compleja (0=No, 1=Si)
    y_directive_gt = torch.randint(0, 2, (batch_size,), dtype=torch.float) # Use float for BCEWithLogitsLoss target


    return x_data, y_token_gt, y_sentiment_gt, y_directive_gt


# --- Función de Recompensa (Ejemplo de Idea 1: Puntuación por Componentes) ---
# Modified to return a tensor loss
def calculate_loss_from_reward(token_logits, sentiment_logits, directive_logits,
                               y_token_gt, y_sentiment_gt, y_directive_gt,
                               rewards_config):
    # Ensure inputs are on the same device if needed, though dummy data is on CPU initially

    # --- Calculate Losses for Each Task ---
    # Using standard differentiable loss functions for each task.
    # The Gumbel-Softmax in the forward pass allows gradients from these losses
    # to flow back to the `self.rl_masks` parameters.

    # Token classification loss (Cross-Entropy)
    # Reshape token_logits to (B*T, num_classes) and y_token_gt to (B*T)
    token_loss = F.cross_entropy(token_logits.view(-1, 4), y_token_gt.view(-1))

    # Sentiment classification loss (Cross-Entropy)
    # sentiment_logits shape (B, num_sentiment_classes), y_sentiment_gt shape (B,)
    sentiment_loss = F.cross_entropy(sentiment_logits, y_sentiment_gt)

    # Directive classification loss (Binary Cross-Entropy with Logits)
    # directive_logits shape (B, 1), y_directive_gt shape (B,)
    # Squeeze directive_logits to (B,) before passing to BCEWithLogitsLoss
    directive_loss = F.binary_cross_entropy_with_logits(directive_logits.squeeze(-1), y_directive_gt)


    # --- Combine Losses ---
    # The total loss is the sum of the individual task losses.
    # You can add weighting factors from rewards_config here if desired
    # (e.g., total_loss = rewards_config['token_weight'] * token_loss + ...),
    # but summing is a common starting point.
    total_loss = token_loss + sentiment_loss + directive_loss

    # If you strictly wanted to use a "reward" function as a scalar loss to maximize,
    # you'd calculate a score based on predictions and ground truth using tensor ops
    # and return -mean(batch_reward). The previous code block's commented-out section
    # showed an attempt at this. However, using standard task losses as shown above
    # is the typical way to leverage the Gumbel-Softmax for differentiable training
    # of parameters that influence categorical choices (like which tokens to attend to via masks).

    # Returning the combined differentiable loss
    return total_loss # This is the loss to minimize


# --- Configuración de Recompensas (used for the alternative RL loss calculation,
# and potentially for weighting losses if implementing that) ---
# Keeping this structure.
rewards_cfg = {
    'token_correct_reward': 1.0,      # Reward for each correct token
    'token_incorrect_penalty': 0.5, # Penalty for each incorrect token
    'sentiment_correct_reward': 10.0, # Reward for correct sentiment
    'sentiment_incorrect_penalty': 5.0, # Penalty for incorrect sentiment
    'directive_correct_reward': 20.0, # Reward for correctly identifying complex directive when present
    'directive_directive_miss_penalty': 25.0, # Penalty for missing a complex directive
    'directive_false_positive_penalty': 10.0 # Penalty for false positive on complex directive
    # Add weights for losses here if you want to weight them
    # 'token_weight': 1.0,
    # 'sentiment_weight': 1.0,
    # 'directive_weight': 1.0,
}

# Parámetros del modelo y entrenamiento
torch.manual_seed(0)
seq_len = 10 # Ahora más largo para simular oraciones
embed_dim = 64 # Dimensión del embedding
num_heads = 4
epochs = 20000 # Más épocas para una tarea más compleja
B = 32       # batch size

model = RevisedMultiRLSoftAttention(embed_dim, num_heads, seq_len)
# Optimizer for standard gradient descent on the combined differentiable loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# We don't need a separate loss_fn if calculate_loss_from_reward returns the loss
# loss_fn = nn.MSELoss() # Not used anymore for this loss calculation method

# Para guardar máscaras en época 1 y final
masks_epoch1 = None
masks_epochFinal = None

print("Iniciando entrenamiento...")
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()

    # Generar datos para el batch actual
    x_data, y_token_gt, y_sentiment_gt, y_directive_gt = generate_dummy_sentence_data(B, seq_len, embed_dim)

    # Move data to device if using GPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    x_data = x_data.to(device)
    y_token_gt = y_token_gt.to(device)
    y_sentiment_gt = y_sentiment_gt.to(device)
    y_directive_gt = y_directive_gt.to(device)
    model.to(device) # Ensure model is on the same device

    # Realizar forward pass
    # output_main: salida del MHA, token_logits: predicciones de token, sentiment_logits: predicciones de sentimiento, directive_logits: predicciones de directiva
    output_main, masks, token_logits, sentiment_logits, directive_logits = model(x_data, training=True)

    # Calcular la pérdida usando la función modificada
    # This function now returns a tensor loss
    loss = calculate_loss_from_reward(
        token_logits,
        sentiment_logits,
        directive_logits,
        y_token_gt,
        y_sentiment_gt,
        y_directive_gt,
        rewards_cfg # Pass rewards config if the function needs it for weighting
    )

    # Backpropagation
    # loss is now a tensor from calculate_loss_from_reward
    loss.backward()
    optimizer.step()

    if epoch == 1:
        # Masks are already tensors on device, clone and move to CPU for storage
        masks_epoch1 = [m.detach().cpu().clone() for m in masks]
    if epoch == epochs:
         # Masks are already tensors on device, clone and move to CPU for storage
         masks_epochFinal = [m.detach().cpu().clone() for m in masks]

    # Print loss (which is a tensor, use .item() for printing scalar)
    if epoch % 1000 == 0 or epoch == 1 or epoch == epochs:
        print(f"Época {epoch}: Loss = {loss.item():.4f}")


# --- Funciones de Ploteo (minor change to move to CPU) ---
def plot_masks(masks, title_prefix=""):
    fig, axs = plt.subplots(1, len(masks), figsize=(16, 4))
    if len(masks) == 1:
        axs = [axs]

    for i, mask in enumerate(masks):
        # Ensure mask is on CPU before converting to numpy
        sns.heatmap(mask.detach().cpu().numpy(), cmap="magma", annot=False, fmt=".2f", ax=axs[i]) # annot=False for less noise
        axs[i].set_title(f"Cabeza {i+1}")
        axs[i].set_xlabel("Tokens que se miran")
        axs[i].set_ylabel("Token que mira")
    plt.suptitle(f"{title_prefix} - Máscaras RL", fontsize=16)
    plt.tight_layout()
    plt.show()

def plot_cosine_similarity(masks, title):
    # Ensure masks are on CPU before converting to numpy
    flat_masks = [m.detach().cpu().flatten().numpy() for m in masks]
    if len(flat_masks) < 2:
         print(f"No hay suficientes cabezas ({len(flat_masks)}) para calcular la similitud del coseno.")
         return
    sim_matrix = cosine_similarity(flat_masks)
    sns.heatmap(sim_matrix, annot=True, cmap="viridis", xticklabels=[f"H{i}" for i in range(len(masks))], yticklabels=[f"H{i}" for i in range(len(masks))])
    plt.title(title)
    plt.show()


# Visualizar resultados
# These variables now hold the masks on CPU
if masks_epoch1 is not None:
    plot_masks(masks_epoch1, "Época 1")
    plot_cosine_similarity(masks_epoch1, "Época 1 - Similitud coseno entre cabezas")

if masks_epochFinal is not None:
    plot_masks(masks_epochFinal, "Época Final")
    plot_cosine_similarity(masks_epochFinal, "Época Final - Similitud coseno entre cabezas")

## Atención suave con RL sobre imágenes

Dos versiones: con tus imágenes (parches) y con MNIST.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


class PatchEmbedding(nn.Module):
    """
    Convierte la imagen en parches, aplana y proyecta a un espacio embebido.
    """
    def __init__(self, patch_size, input_dim, embedding_dim):
        super().__init__()
        self.patch_size = patch_size
        self.input_dim = input_dim  # Canales de la imagen (e.g., 1 para MNIST)
        self.embedding_dim = embedding_dim
        self.projection_layer = nn.Linear(patch_size * patch_size * input_dim, embedding_dim)

    def forward(self, x):
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        assert H % self.patch_size == 0 and W % self.patch_size == 0, "La imagen debe ser divisible por patch_size"

        # Extraer patches
        patches = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
        # patches shape: (B, C, num_patches_H, num_patches_W, patch_size, patch_size)

        num_patches_H = patches.shape[2]
        num_patches_W = patches.shape[3]
        num_patches = num_patches_H * num_patches_W

        # Rearreglar para que sea (B, num_patches, C, patch_size, patch_size)
        patches = patches.contiguous().view(B, C, num_patches, self.patch_size, self.patch_size)
        patches = patches.permute(0, 2, 1, 3, 4).contiguous()  # (B, num_patches, C, patch_size, patch_size)

        # Aplanar cada patch
        patches = patches.view(B, num_patches, -1)  # (B, num_patches, patch_size*patch_size*C)

        # Proyectar cada patch
        embeddings = self.projection_layer(patches)  # (B, num_patches, embedding_dim)
        return embeddings


class RevisedMultiRLSoftAttentionHead(nn.Module):
    """
    Cabeza individual de atención relajada con logits entrenables para pesos de atención.
    """
    def __init__(self, input_dim, head_dim, seq_len):
        super().__init__()
        self.input_dim = input_dim
        self.head_dim = head_dim
        self.seq_len = seq_len

        # Proyecciones para queries, keys, values
        self.query_layer = nn.Linear(input_dim, head_dim)
        self.key_layer = nn.Linear(input_dim, head_dim)
        self.value_layer = nn.Linear(input_dim, head_dim)

        # Parámetro entrenable para logits de atención (sin softmax directo)
        self.attn_logits = nn.Parameter(torch.randn(seq_len, seq_len))

        # Temperatura para suavizado
        self.temperature = 1.0

    def forward(self, x, training=False):
        # x: (B, T, input_dim)
        B, T, _ = x.shape

        Q = self.query_layer(x)  # (B, T, head_dim)
        K = self.key_layer(x)    # (B, T, head_dim)
        V = self.value_layer(x)  # (B, T, head_dim)

        # Scores de atención: QK^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, T, T)

        # Crear máscara: 1 en diagonal y superior, 0 debajo para atención causal (opcional)
        attn_mask = torch.tril(torch.ones(T, T), diagonal=0).to(x.device)  # (T, T)

        # Aplicar máscara sumando -inf donde hay 0 para que softmax no atienda esas posiciones
        mask = (attn_mask == 0)
        scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))

        # Añadir logits entrenables (expandir para batch)
        scores = scores + self.attn_logits.unsqueeze(0)

        # Softmax con temperatura
        attn_weights = F.softmax(scores / self.temperature, dim=-1)  # (B, T, T)

        # Aplicar atención
        out = torch.matmul(attn_weights, V)  # (B, T, head_dim)

        return out, attn_weights


class RevisedMultiRLSoftAttention(nn.Module):
    """
    Multi-Head Attention con atención relajada y parámetros entrenables de logits.
    """
    def __init__(self, input_dim, head_dim, num_heads, seq_len):
        super().__init__()
        self.input_dim = input_dim
        self.head_dim = head_dim
        self.num_heads = num_heads
        self.seq_len = seq_len

        assert head_dim * num_heads == input_dim, "input_dim debe ser num_heads * head_dim"

        # Crear heads independientes
        self.heads = nn.ModuleList(
            [RevisedMultiRLSoftAttentionHead(input_dim, head_dim, seq_len) for _ in range(num_heads)]
        )

        # Proyección final
        self.out_projection = nn.Linear(num_heads * head_dim, input_dim)

    def forward(self, x, training=False):
        # x: (B, T, input_dim)
        B, T, D = x.shape
        head_outputs = []
        attn_weights_all = []

        for head in self.heads:
            out, attn_weights = head(x, training=training)
            head_outputs.append(out)
            attn_weights_all.append(attn_weights)

        # Concatenar outputs de todas las cabezas
        concat_out = torch.cat(head_outputs, dim=-1)  # (B, T, num_heads*head_dim)
        out = self.out_projection(concat_out)         # (B, T, input_dim)

        # Promedio de matrices de atención para visualización
        avg_attn_weights = torch.stack(attn_weights_all).mean(dim=0)  # (B, T, T)

        return out, avg_attn_weights


def visualize_attention_map(attn_weights, patch_size, image_size):
    """
    Visualiza el mapa de atención promedio en escala de grises.
    attn_weights: tensor (T, T) para un solo batch
    patch_size: tamaño del parche
    image_size: (H, W) tamaño original imagen
    """
    import numpy as np

    num_patches_H = image_size[0] // patch_size
    num_patches_W = image_size[1] // patch_size
    num_patches = num_patches_H * num_patches_W

    attn_weights = attn_weights.detach().cpu().numpy()  # (T, T)
    if attn_weights.shape[0] != num_patches:
        print(f"Warning: attn_weights shape {attn_weights.shape} no coincide con número de parches {num_patches}")
        return

    # Promedio la atención de cada parche hacia los demás (suma o media de filas)
    attn_map_1d = attn_weights.mean(axis=0)  # Promedio atención hacia cada parche

    # Reorganizar 1D a 2D según disposición de parches
    attn_map_2d = attn_map_1d.reshape(num_patches_H, num_patches_W)

    plt.imshow(attn_map_2d, cmap='gray')
    plt.title('Mapa de atención promedio por parche')
    plt.colorbar()
    plt.show()


# Ejemplo de uso básico

if __name__ == "__main__":
    # Parámetros
    B = 2
    C = 1
    H, W = 28, 28
    patch_size = 7
    embedding_dim = 64
    num_heads = 4
    head_dim = embedding_dim // num_heads
    seq_len = (H // patch_size) * (W // patch_size)

    # Imagen aleatoria
    x = torch.randn(B, C, H, W)

    # Patch embedding
    patch_embed = PatchEmbedding(patch_size, C, embedding_dim)
    embeddings = patch_embed(x)  # (B, num_patches, embedding_dim)

    # Multihead attention
    attention = RevisedMultiRLSoftAttention(embedding_dim, head_dim, num_heads, seq_len)
    out, attn_weights = attention(embeddings)

    print(f"Output shape: {out.shape}")
    print(f"Attention weights shape: {attn_weights.shape}")

    # Visualizar atención para batch 0
    visualize_attention_map(attn_weights[0], patch_size, (H, W))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import random

# Definición de las clases RLSoftAttentionHead y RevisedMultiRLSoftAttention
# (son exactamente las mismas que en tu código anterior, no las repito aquí para brevedad)

class RLSoftAttentionHead(nn.Module):
    def __init__(self, embed_dim, seq_len, temperature=0.7):
        super().__init__()
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.seq_len = seq_len
        self.temperature = temperature
        self.attn_logits = nn.Parameter(torch.randn(seq_len, seq_len))

    def forward(self, x, training=True, forced_attention_mask=None):
        B, T, E = x.shape
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)
        d_k = Q.shape[-1]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)

        if training:
            noise = -torch.log(-torch.log(torch.rand_like(self.attn_logits)))
            mask_logits = (self.attn_logits + noise) / self.temperature
        else:
            mask_logits = self.attn_logits

        attn_mask = F.softmax(mask_logits, dim=-1)

        if forced_attention_mask is not None:
            forced_attention_mask = forced_attention_mask.to(x.device)
            # Changed from scores * forced_attention_mask to scores + forced_attention_mask
            # as suggested by the original ipython-input-26 code
            masked_scores = scores + forced_attention_mask.unsqueeze(0).expand(B, -1, -1)
        else:
            # If no forced mask, use the learned attention mask from attn_logits
            attn_mask_exp = attn_mask.unsqueeze(0).expand(B, -1, -1)
            masked_scores = scores * attn_mask_exp # Keep multiplication here for learned mask

        attn_weights = F.softmax(masked_scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        return output, attn_weights, attn_mask

class RevisedMultiRLSoftAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, seq_len):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList([
            RLSoftAttentionHead(self.head_dim, seq_len) for _ in range(num_heads)
        ])
        self.output_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, training=True, forced_attention_masks=None):
        B, T, E = x.shape
        if E != self.num_heads * self.head_dim:
             raise ValueError(f"Input embedding dimension mismatch. Expected {self.num_heads * self.head_dim}, got {E}")

        x_split = x.view(B, T, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        x_split = x_split.contiguous().view(B * self.num_heads, T, self.head_dim)
        outputs = []
        attention_masks = []
        for i, head in enumerate(self.heads):
            current_forced_mask = forced_attention_masks[i] if forced_attention_masks and i < len(forced_attention_masks) and forced_attention_masks[i] is not None else None # Add check for None mask
            # Need to pass the correct slice for the head
            head_output, head_attn_weights, head_attn_mask = head(x_split[i*B:(i+1)*B], training=training, forced_attention_mask=current_forced_mask)
            outputs.append(head_output)
            attention_masks.append(head_attn_mask)
        combined_outputs = torch.cat([o.view(B, T, self.head_dim) for o in outputs], dim=-1)
        output = self.output_proj(combined_outputs)
        return output, attention_masks

image_path = '/content/H.jpg'
patch_size = 16

NORM_DIM = 64
NUM_ATTENTION_HEADS = 2
CONTOUR_PIXELS = 16 # Esto define el radio de la región de atención forzada

try:
    image = Image.open(image_path).convert('RGB')
except FileNotFoundError:
    print(f"Error: La imagen no se encontró en la ruta: {image_path}")
    raise FileNotFoundError(f"Error: La imagen no se encontró en la ruta: {image_path}")

image = image.resize((NORM_DIM, NORM_DIM), Image.LANCZOS) # Normaliza la imagen

img_width, img_height = image.size
n_patches_width = img_width // patch_size
n_patches_height = img_height // patch_size
num_patches = n_patches_width * n_patches_height
seq_len = num_patches

print(f"Imagen normalizada a: {img_width}x{img_height}")
print(f"Tamaño del parche: {patch_size}x{patch_size}")
print(f"Número de parches (ancho x alto): {n_patches_width} x {n_patches_height} = {num_patches}")

def extract_patches(img, patch_size):
    patches = []
    width, height = img.size
    for i in range(0, height, patch_size):
        for j in range(0, width, patch_size):
            box = (j, i, j + patch_size, i + patch_size)
            patch = img.crop(box)
            patches.append(patch)
    return patches

patches = extract_patches(image, patch_size)
transform = transforms.ToTensor()
patch_tensors = [transform(patch) for patch in patches]
patch_tensor_stack = torch.stack(patch_tensors)

embedding_dim = 64 # Mantengamos un embed_dim fijo y razonable
input_flat_dim = patch_size * patch_size * patch_tensor_stack.shape[1]

projection_layer = torch.nn.Linear(input_flat_dim, embedding_dim)
patch_embeddings = patch_tensor_stack.view(num_patches, -1)
patch_embeddings = projection_layer(patch_embeddings)

batch_size = 1
input_sequence = patch_embeddings.unsqueeze(0)

print("Forma del tensor de entrada para el modelo:", input_sequence.shape)

model = RevisedMultiRLSoftAttention(embed_dim=embedding_dim, num_heads=NUM_ATTENTION_HEADS, seq_len=seq_len)

def get_attention_masks_and_info(num_heads, img_width, img_height, patch_size, n_patches_width, n_patches_height, seq_len, contour_pixels, current_points):
    """
    Genera las máscaras de atención forzadas y la información de la cabeza.
    current_points: Lista de (x, y) actuales para cada cabeza.
    Retorna: forced_masks_for_all_heads, head_positions_info, points_used
    """
    points_used = list(current_points) # Use the provided current_points as the points_used for this iteration

    forced_masks_for_all_heads = []
    # Usamos un conjunto para asegurar que los parches asignados sean únicos en esta iteración.
    assigned_patch_indices_this_iter = set()
    head_positions_info = []

    for head_idx, (target_x, target_y) in enumerate(points_used): # Iterate using points_used
        region_patch_indices = []
        for i in range(n_patches_height):
            for j in range(n_patches_width):
                patch_center_x = j * patch_size + patch_size // 2
                patch_center_y = i * patch_size + patch_size // 2
                distance = np.sqrt((patch_center_x - target_x)**2 + (patch_center_y - target_y)**2)

                # Si el parche está dentro del contorno y aún no ha sido asignado a otra cabeza en esta iteración
                if distance <= contour_pixels and (i * n_patches_width + j) not in assigned_patch_indices_this_iter:
                    region_patch_indices.append(i * n_patches_width + j)

        if not region_patch_indices:
            print(f"Advertencia: Cabeza {head_idx+1}: Ningún parche único encontrado para ({target_x}, {target_y}). La máscara no se forzará para esta cabeza.")
            forced_masks_for_all_heads.append(None)
            head_positions_info.append(f"Cabeza {head_idx+1} en posición X:{target_x}, Y:{target_y} (sin parches únicos asignados)")
        else:
            forced_attn_mask_tensor = torch.full((seq_len, seq_len), -1e9) # Penaliza todo
            for idx_from in region_patch_indices:
                for idx_to in region_patch_indices:
                    forced_attn_mask_tensor[idx_from, idx_to] = 0.0 # Permite atención dentro de la región
            forced_masks_for_all_heads.append(forced_attn_mask_tensor)
            assigned_patch_indices_this_iter.update(region_patch_indices) # Marca estos parches como "usados"
            head_positions_info.append(f"Cabeza {head_idx+1} en posición X:{target_x}, Y:{target_y}")

    return forced_masks_for_all_heads, head_positions_info, points_used # Return points_used

def visualize_attention_cpa(image, attention_mask, patch_size, head_idx, target_coords, assigned_regions, radius, head_info_string):
    n_patches_width = image.width // patch_size
    n_patches_height = image.height // patch_size

    if assigned_regions:
        closest_patch_idx = -1
        min_dist = float('inf')
        for p_idx in assigned_regions:
            p_row = p_idx // n_patches_width
            p_col = p_idx % n_patches_width
            p_center_x = p_col * patch_size + patch_size // 2
            p_center_y = p_row * patch_size + patch_size // 2
            dist = np.sqrt((p_center_x - target_coords[0])**2 + (p_center_y - target_coords[1])**2)
            if dist < min_dist:
                min_dist = dist
                closest_patch_idx = p_idx
        # Ensure closest_patch_idx is valid before indexing
        if closest_patch_idx != -1 and closest_patch_idx < attention_mask.size(0):
             attention_from_target = attention_mask.cpu().numpy()[closest_patch_idx]
             title_suffix = f'desde parche más cercano a ({target_coords[0]}, {target_coords[1]})'
        else:
            # Fallback if the determined closest patch index is invalid
            attention_from_target = attention_mask.cpu().numpy()[0]
            title_suffix = 'desde el primer parche (error en índice del parche más cercano)'

    else:
        # Fallback if no region was assigned
        attention_from_target = attention_mask.cpu().numpy()[0]
        title_suffix = 'desde el primer parche (sin región asignada)'


    attention_grid = attention_from_target.reshape(n_patches_height, n_patches_width)

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(image)
    ax.axis('off')

    heatmap = ax.imshow(attention_grid, cmap="viridis", alpha=0.7,
                        extent=[0, image.width, image.height, 0], origin='upper')

    cbar = fig.colorbar(heatmap, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)
    cbar.set_label(f'Peso de Atención ({title_suffix})')

    circle = plt.Circle((target_coords[0], target_coords[1]), radius, color='red', fill=False, linewidth=3, linestyle='--')
    ax.add_patch(circle)

    for p_idx in assigned_regions:
        p_row = p_idx // n_patches_width
        p_col = p_idx % n_patches_width
        rect = plt.Rectangle((p_col * patch_size, p_row * patch_size), patch_size, patch_size,
                             linewidth=1, edgecolor='blue', facecolor='none')
        ax.add_patch(rect)

    plt.title(f'Atención Guiada por Posición (CPA) - {head_info_string}')
    plt.show()

# --- Simulación del Agente RL y el Bucle de Entrenamiento ---

# Inicializa las posiciones de las cabezas (ej. aleatorio al principio)
# Use a different variable name than current_head_positions initially
initial_head_positions = [(random.randint(0, img_width - 1), random.randint(0, img_height - 1)) for _ in range(NUM_ATTENTION_HEADS)]

# Start the loop with the initial positions
current_head_positions = list(initial_head_positions)


max_iterations = 5 # Límite de iteraciones para el ciclo finito
iteration = 0

print("\n--- ¡Iniciando el ciclo de RL manual! ---")
print("El modelo intentará ajustar la atención en base a tu feedback.")
print("Tu tarea es evaluar la 'coherencia' de la atención mostrada.")
print("0: Coherente (buena atención), 1: Incoherente (mala atención)")
print("Presiona Enter sin valor para usar la heurística de ajuste.")
print("-" * 50)

while iteration < max_iterations:
    iteration += 1
    print(f"\n--- Iteración {iteration} ---")

    # Obtener máscaras forzadas y la información de la cabeza para las posiciones actuales
    # The get_attention_masks_and_info function now returns the list of points used
    forced_masks_for_all_heads, head_positions_info, points_used_in_iter = get_attention_masks_and_info(
        NUM_ATTENTION_HEADS, img_width, img_height, patch_size, n_patches_width, n_patches_height,
        seq_len, CONTOUR_PIXELS, current_head_positions # Pass current_head_positions
    )

    print("\nPosiciones de las Cabezas de Atención en esta iteración:")
    for info in head_positions_info:
        print(info)

    # Realizar el forward pass del modelo
    model.eval() # Modo de evaluación
    with torch.no_grad():
        output, attention_masks = model(input_sequence, forced_attention_masks=forced_masks_for_all_heads)

    # Visualizar la atención y pedir feedback al usuario
    for head_idx in range(NUM_ATTENTION_HEADS):
        # We need the region indices based on the *actual points used in this iteration*
        # Recalculate the region_patch_indices for visualization based on points_used_in_iter
        current_target_point = points_used_in_iter[head_idx]
        current_region_indices = []
        for i in range(n_patches_height):
            for j in range(n_patches_width):
                patch_center_x = j * patch_size + patch_size // 2
                patch_center_y = i * patch_size + patch_size // 2
                distance = np.sqrt((patch_center_x - current_target_point[0])**2 + (patch_center_y - current_target_point[1])**2)
                if distance <= CONTOUR_PIXELS:
                    # We don't need to check for uniqueness here, just identify the patches within the contour
                    current_region_indices.append(i * n_patches_width + j)


        # Check if there were any patches assigned for this head's mask *during mask generation*
        # The visualize function needs the actual region indices used to generate the mask
        # This requires a small modification to get_attention_masks_and_info to return the region indices per head
        # For now, let's assume forced_masks_for_all_heads[head_idx] is None if no unique patches were found,
        # and visualize only if it's not None. The `current_region_indices` calculated above is
        # based on the target point and CONTOUR_PIXELS, which is what visualize_attention_cpa expects for plotting the circle and rects.
        # The `assigned_regions` argument to visualize_attention_cpa should ideally be the list of patch indices
        # that were *actually assigned* to this head based on the uniqueness logic in get_attention_masks_and_info.

        # Temporarily, let's pass the calculated current_region_indices for visualization,
        # keeping in mind this might include patches that were "stolen" by earlier heads
        # in the mask generation if not handled perfectly. A proper fix would involve
        # returning the exact assigned_patch_indices_this_iter per head from get_attention_masks_and_info.

        # For simplicity and fixing the NameError, let's proceed with the calculated current_region_indices
        # for visualization, which are based on the target point and contour, not the uniqueness assignment.
        # This visualization might show patches that were *eligible* but not *assigned* for mask generation.

        # Check if the mask for this head is None (meaning no unique patches were assigned)
        mask_is_none = forced_masks_for_all_heads[head_idx] is None

        if not mask_is_none: # Only visualize if a mask was actually generated for this head
             visualize_attention_cpa(image, attention_masks[head_idx], patch_size, head_idx,
                                     current_target_point, current_region_indices, CONTOUR_PIXELS, head_positions_info[head_idx])
        else:
             print(f"No se visualiza la Cabeza {head_idx+1} en esta iteración (no se asignaron parches únicos a su región).")


    # --- SIMULACIÓN DEL AGENTE RL ---
    # Aquí es donde el "agente" (simulado manualmente o por una heurística simple)
    # ajustaría las coordenadas basándose en la retroalimentación.

    feedback = input(f"Evalúa la coherencia de la atención para esta iteración (0: Coherente, 1: Incoherente, o presiona Enter para ajustar automáticamente): ")

    new_head_positions = list(current_head_positions) # Copia para modificar

    if feedback == '0':
        print("¡Excelente! La atención es coherente. Intentaremos mantener posiciones similares.")
        # Pequeño ajuste aleatorio para explorar un poco, pero no moverse mucho
        for i in range(NUM_ATTENTION_HEADS):
            x, y = new_head_positions[i]
            x_adj = max(0, min(img_width - 1, x + random.randint(-5, 5)))
            y_adj = max(0, min(img_height - 1, y + random.randint(-5, 5)))
            new_head_positions[i] = (x_adj, y_adj)
    elif feedback == '1':
        print("¡Incoherente! El modelo intentará moverse a nuevas posiciones.")
        # Mover las posiciones aleatoriamente para explorar nuevas áreas
        for i in range(NUM_ATTENTION_HEADS):
            x_new = random.randint(0, img_width - 1)
            y_new = random.randint(0, img_height - 1)
            new_head_positions[i] = (x_new, y_new)
    elif feedback == '':
        print("Ajuste automático: Movimiento aleatorio simple para explorar.")
        # Si el usuario no da feedback específico, ajusta aleatoriamente para seguir el ciclo
        for i in range(NUM_ATTENTION_HEADS):
            x_new = random.randint(0, img_width - 1)
            y_new = random.randint(0, img_height - 1)
            new_head_positions[i] = (x_new, y_new)
    else:
        print("Entrada no válida. Usando ajuste automático por defecto.")
        for i in range(NUM_ATTENTION_HEADS):
            x_new = random.randint(0, img_width - 1)
            y_new = random.randint(0, img_height - 1)
            new_head_positions[i] = (x_new, y_new)

    current_head_positions = new_head_positions # Update positions for the next iteration

    if iteration == max_iterations:
        print("\nFin del ciclo de RL simulado.")
        break

    cont_input = input("¿Continuar a la siguiente iteración (s/n)? ").lower()
    if cont_input != 's':
        print("Saliendo del ciclo.")
        break

## Atención a la salida (AttentiveLinear)

En el banco: `atencion_salida`. Ojo: los «factores de atención» son una matriz fija (no dependen de la entrada), así que la capa equivale a una lineal con pesos entre 0 y 1.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Hiperparámetros Globales
HIPERPARAMETROS = {
    "num_neuronas_ocultas_fc1": 128,
    "num_neuronas_ocultas_attentive": 64, # This should be the output size of the attentive layer and input size of the next layer (fc3)
    "num_clases_salida": 10, # This is the output size of the final layer (fc3)
    "learning_rate": 0.001,
    "epochs": 5,
    "batch_size_train": 64,
    "batch_size_test": 1000,
    "paciencia_early_stop": 3,
    "factor_reduccion_lr": 0.5,
    "min_lr": 1e-6
}

# Capa Lineal con Atención a la Salida
class AttentiveLinear(nn.Module):
    def __init__(self, in_features, out_features, next_layer_in_features):
        """
        in_features: Input features from the previous layer (fc1 output size).
        out_features: Output features of this layer (input size for fc3).
        next_layer_in_features: Output size of the *next* layer (fc3 output size, number of classes).
                                This is used to define the shape of attention_weights.
        """
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.next_layer_in_features = next_layer_in_features # This parameter is used for attention_weights shape

        # Standard linear layer weights and bias
        self.linear_weights = nn.Parameter(torch.Tensor(out_features, in_features))
        self.linear_bias = nn.Parameter(torch.Tensor(out_features))

        # Attention weights connecting output features to the number of final classes
        # Shape (out_features, next_layer_in_features)
        self.attention_weights = nn.Parameter(torch.Tensor(out_features, next_layer_in_features))

        self._initialize_parameters()

    def _initialize_parameters(self):
        # Initialize linear layer weights and bias (standard Kaiming uniform for ReLU)
        nn.init.kaiming_uniform_(self.linear_weights, mode='fan_in', nonlinearity='relu')
        fan_in_bias = self.linear_weights.size(1)
        bound_bias = 1 / np.sqrt(fan_in_bias)
        nn.init.uniform_(self.linear_bias, -bound_bias, bound_bias)

        # Initialize attention weights (Xavier uniform is a common choice)
        nn.init.xavier_uniform_(self.attention_weights)

    def forward(self, x):
        # Step 1: Perform the standard linear transformation
        # Input x shape: (batch_size, in_features) -> (64, 128)
        # linear_weights shape: (out_features, in_features) -> (64, 128)
        # Output linear_raw shape: (batch_size, out_features) -> (64, 64)
        output_linear_raw = F.linear(x, self.linear_weights, self.linear_bias)

        # Step 2: Apply activation (ReLU) to the linear output
        # activations shape: (batch_size, out_features) -> (64, 64)
        activations = F.relu(output_linear_raw)

        # Step 3: Apply sigmoid to attention weights to get factors between 0 and 1
        # attention_weights shape: (out_features, next_layer_in_features) -> (64, 10)
        attention_factors = torch.sigmoid(self.attention_weights) # Shape: (out_features, next_layer_in_features) -> (64, 10)

        # Step 4: Reshape activations and attention_factors for broadcasting
        # activations: (batch_size, out_features) -> (64, 64) -> unsqueeze(2) -> (64, 64, 1)
        # attention_factors: (out_features, next_layer_in_features) -> (64, 10) -> unsqueeze(0) -> (1, 64, 10)

        # Step 5: Element-wise multiplication after broadcasting.
        # The shapes broadcast to (batch_size, out_features, next_layer_in_features)
        # (64, 64, 1) * (1, 64, 10) = (64, 64, 10)
        attended_output_components = activations.unsqueeze(2) * attention_factors.unsqueeze(0)

        # Step 6: Sum over the `next_layer_in_features` dimension (dim=2).
        # This dimension has size 10. Summing over it reduces the shape from (64, 64, 10) to (64, 64).
        # The resulting shape (batch_size, out_features) is correct for the next linear layer (fc3).
        final_output = attended_output_components.sum(dim=2) # Changed dim=1 to dim=2

        return final_output

# MLP Estándar
class SimpleMLPStandard(nn.Module):
    def __init__(self):
        super().__init__()
        # Input layer: 28*28 flattened image pixels
        self.fc1 = nn.Linear(28 * 28, HIPERPARAMETROS["num_neuronas_ocultas_fc1"])
        # Hidden layer: connects fc1 output to fc2 input
        self.fc2 = nn.Linear(HIPERPARAMETROS["num_neuronas_ocultas_fc1"], HIPERPARAMETROS["num_neuronas_ocultas_attentive"])
        # Output layer: connects fc2 output to the number of classes
        self.fc3 = nn.Linear(HIPERPARAMETROS["num_neuronas_ocultas_attentive"], HIPERPARAMETROS["num_clases_salida"])

    def forward(self, x):
        # Flatten the input image
        x = x.view(-1, 28 * 28)
        # Pass through the first linear layer and apply ReLU activation
        x = F.relu(self.fc1(x))
        # Pass through the second linear layer and apply ReLU activation
        x = F.relu(self.fc2(x))
        # Pass through the final linear layer (no activation here, as CrossEntropyLoss expects raw scores)
        x = self.fc3(x)
        return x

# MLP con Capa de Atención de Salida (replacing fc2)
class SimpleMLPOutputAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # Input layer: 28*28 flattened image pixels
        self.fc1 = nn.Linear(28 * 28, HIPERPARAMETROS["num_neuronas_ocultas_fc1"])

        # Attentive layer: replaces the standard fc2 layer
        # It takes input from fc1, has its own 'out_features' (matching fc3's input size),
        # and uses 'next_layer_in_features' (matching fc3's output size) for attention weights.
        self.attentive_layer = AttentiveLinear(
            in_features=HIPERPARAMETROS["num_neuronas_ocultas_fc1"], # Input size is the output size of fc1
            out_features=HIPERPARAMETROS["num_neuronas_ocultas_attentive"], # Output size for this layer, input size for fc3
            next_layer_in_features=HIPERPARAMETROS["num_clases_salida"] # Output size of the *next* layer (fc3)
        )

        # Output layer: takes input from the attentive layer and outputs to the number of classes
        self.fc3 = nn.Linear(HIPERPARAMETROS["num_neuronas_ocultas_attentive"], HIPERPARAMETROS["num_clases_salida"])

    def forward(self, x):
        # Flatten the input image
        x = x.view(-1, 28 * 28)
        # Pass through the first linear layer and apply ReLU activation
        x = F.relu(self.fc1(x))
        # Pass through the attentive layer. The attentive layer applies ReLU internally.
        x = self.attentive_layer(x)
        # Pass through the final linear layer (no activation here)
        x = self.fc3(x)
        return x

# Preparación de Datos
def get_mnist_data_loaders(batch_size_train, batch_size_test):
    """
    Downloads and prepares MNIST data loaders.
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    # Download MNIST dataset
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size_train, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size_test, shuffle=False)
    return train_loader, test_loader

# Función de Entrenamiento y Evaluación
def train_and_evaluate(model, train_loader, test_loader, optimizer, criterion, params, model_name="Modelo"):
    """
    Trains the model for a specified number of epochs and evaluates its performance.
    Includes Early Stopping and ReduceLROnPlateau scheduler.
    """
    print(f"\n--- Entrenando {model_name} ---")

    # Set the model to training mode
    model.train()

    best_accuracy = -1 # Track best accuracy for Early Stopping
    epochs_no_improve = 0 # Counter for Early Stopping

    # Initialize the scheduler
    # Reduce LR when the validation accuracy stops improving
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max', # Monitor maximum accuracy
        factor=params["factor_reduccion_lr"], # Factor by which the learning rate will be reduced
        patience=params["paciencia_early_stop"] // 2, # Number of epochs with no improvement after which learning rate will be reduced
        min_lr=params["min_lr"], # Minimum learning rate
        verbose=True # Print message when LR is reduced
    )

    for epoch in range(1, params["epochs"] + 1):
        total_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            # Zero out the gradients from the previous step
            optimizer.zero_grad()

            # Forward pass: compute predicted outputs by passing inputs to the model
            output = model(data)

            # Calculate the batch loss
            loss = criterion(output, target)

            # Backward pass: compute gradient of the loss with respect to model parameters
            loss.backward()

            # Perform a single optimization step (parameter update)
            optimizer.step()

            total_loss += loss.item() * data.size(0) # Accumulate loss, weighted by batch size

            # Print training status periodically
            if batch_idx % 100 == 0:
                print(f'{model_name} - Época {epoch}, Lote {batch_idx}/{len(train_loader)}, Pérdida: {loss.item():.6f}')

        avg_train_loss = total_loss / len(train_loader.dataset)
        print(f'{model_name} - Época {epoch}, Pérdida Promedio Entrenamiento: {avg_train_loss:.4f}')


        # Evaluation at the end of each epoch for Early Stopping and Scheduler
        # Use silent=True so evaluate_model doesn't print batch details during training loop evaluation
        current_accuracy = evaluate_model(model, test_loader, criterion, silent=True)
        print(f'{model_name} - Época {epoch}, Precisión en Test: {current_accuracy:.2f}%')

        # Check for Early Stopping
        if current_accuracy > best_accuracy:
            best_accuracy = current_accuracy
            epochs_no_improve = 0
            # Here you might save the model state if accuracy improves significantly
            # Example: torch.save(model.state_dict(), f"{model_name.replace(' ', '_')}_best_acc.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= params["paciencia_early_stop"]:
                print(f"Early stopping en Época {epoch}. Precisión no mejoró por {params['paciencia_early_stop']} épocas.")
                break

        # Step the scheduler based on the test accuracy
        # This will reduce LR if accuracy hasn't improved for 'patience' epochs defined in scheduler
        scheduler.step(current_accuracy)

        # Check if LR has dropped below the minimum after scheduler step
        if optimizer.param_groups[0]['lr'] < params["min_lr"]:
            print(f"Learning Rate ha caído por debajo del mínimo en Época {epoch}.")
            break

    print(f"\n--- Evaluación Final {model_name} ---")
    # Final evaluation, print full details
    final_accuracy = evaluate_model(model, test_loader, criterion)
    return final_accuracy

def evaluate_model(model, test_loader, criterion, silent=False):
    """
    Evaluates the model's performance on the test set.
    """
    # Set the model to evaluation mode
    model.eval()
    test_loss = 0
    correct = 0
    # Turn off gradient calculations for evaluation
    with torch.no_grad():
        for data, target in test_loader:
            # Compute predicted outputs
            output = model(data)
            # Calculate the batch loss and add to total
            # We multiply by data.size(0) to get the sum of losses in the batch
            test_loss += criterion(output, target).item() * data.size(0)
            # Get the index of the max log-probability (the predicted class)
            pred = output.argmax(dim=1, keepdim=True)
            # Count correct predictions
            correct += pred.eq(target.view_as(pred)).sum().item()

    # Compute average test loss
    test_loss /= len(test_loader.dataset)
    # Compute accuracy
    accuracy = 100. * correct / len(test_loader.dataset)

    # Print results if not silent
    if not silent:
        print(f'Pérdida de Prueba Promedio: {test_loss:.4f}, Precisión: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)')

    return accuracy

# Ejecución Principal
if __name__ == "__main__":
    # Get data loaders for MNIST
    train_loader, test_loader = get_mnist_data_loaders(
        HIPERPARAMETROS["batch_size_train"],
        HIPERPARAMETROS["batch_size_test"]
    )

    # Initialize both models
    mlp_standard = SimpleMLPStandard()
    mlp_attention = SimpleMLPOutputAttention()

    # Initialize optimizers for both models using Adam
    optimizer_standard = optim.Adam(mlp_standard.parameters(), lr=HIPERPARAMETROS["learning_rate"])
    optimizer_attention = optim.Adam(mlp_attention.parameters(), lr=HIPERPARAMETROS["learning_rate"])

    # Define the loss function (Cross Entropy Loss for classification)
    criterion = nn.CrossEntropyLoss()

    # Train and evaluate the standard MLP
    acc_standard = train_and_evaluate(
        mlp_standard, train_loader, test_loader, optimizer_standard, criterion,
        params=HIPERPARAMETROS,
        model_name="MLP Estándar"
    )

    # Train and evaluate the MLP with the attentive layer
    acc_attention = train_and_evaluate(
        mlp_attention, train_loader, test_loader, optimizer_attention, criterion,
        params=HIPERPARAMETROS,
        model_name="MLP con Atención de Salida"
    )

    # Print final comparison
    print("\n--- Comparación Final de Precisión ---")
    print(f"MLP Estándar: {acc_standard:.2f}%")
    print(f"MLP con Atención de Salida: {acc_attention:.2f}%")

## Atención local en series sinusoidales

En el banco: `atencion_local`.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# --- 1. Generación del Dataset de Onda Sinusoidal ---
def generar_serie_sinusoidal(n_puntos, ventana_entrada):
    x = np.linspace(0, 20 * np.pi, n_puntos)
    y = np.sin(x) + np.random.normal(0, 0.1, n_puntos) # Onda sinusoidal con algo de ruido

    X, Y = [], []
    for i in range(len(y) - ventana_entrada):
        X.append(y[i:(i + ventana_entrada)])
        Y.append(y[i + ventana_entrada])
    return np.array(X), np.array(Y)

N_PUNTOS = 1000
VENTANA_ENTRADA = 10 # Usar 10 puntos anteriores para predecir el siguiente

X_data, Y_data = generar_serie_sinusoidal(N_PUNTOS, VENTANA_ENTRADA)

# Dividir en conjuntos de entrenamiento y prueba
split_ratio = 0.8
split_index = int(N_PUNTOS * split_ratio)

X_train, Y_train = X_data[:split_index], Y_data[:split_index]
X_test, Y_test = X_data[split_index:], Y_data[split_index:]

# --- 2. Los Modelos ---

# Modelo con "Neurona Común" (SGD Básico)
model_sgd = tf.keras.Sequential([
    # Una capa densa que procesa la ventana de entrada
    tf.keras.layers.Dense(32, activation='relu', input_shape=(VENTANA_ENTRADA,)),
    # Una sola neurona de salida para predecir el siguiente valor
    tf.keras.layers.Dense(1)
])
model_sgd.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01), loss='mse')
print("--- Modelo SGD ---")
model_sgd.summary()

# Modelo con "Historial Mejorado" (SGD experimental)

model_sgdm = tf.keras.Sequential([
    # Una capa densa que procesa la ventana de entrada
    tf.keras.layers.Dense(32, activation='relu', input_shape=(VENTANA_ENTRADA,)),
    # Una sola neurona de salida para predecir el siguiente valor
    tf.keras.layers.Dense(1)
])
model_sgd.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01), loss='mse')
print("--- Modelo SGD ---")
model_sgd.summary()

# --- 3. Entrenamiento y Evaluación ---

EPOCHS = 50 # Número de veces que el modelo verá todo el dataset

print("\nEntrenando Modelo SGD...")
history_sgd = model_sgd.fit(X_train, Y_train, epochs=EPOCHS, verbose=0, validation_data=(X_test, Y_test))

print("Entrenando Modelo Adam...")
history_adam = model_adam.fit(X_train, Y_train, epochs=EPOCHS, verbose=0, validation_data=(X_test, Y_test))

# Evaluación de rendimiento en el conjunto de prueba
loss_sgd = model_sgd.evaluate(X_test, Y_test, verbose=0)
loss_adam = model_adam.evaluate(X_test, Y_test, verbose=0)

print(f"\n--- Resultados ---")
print(f"MSE del Modelo SGD en datos de prueba: {loss_sgd:.4f}")
print(f"MSE del Modelo Adam en datos de prueba: {loss_adam:.4f}")

# Visualización del aprendizaje (pérdida en el tiempo)
plt.figure(figsize=(12, 6))
plt.plot(history_sgd.history['loss'], label='SGD Train Loss')
plt.plot(history_sgd.history['val_loss'], label='SGD Validation Loss')
plt.plot(history_adam.history['loss'], label='Adam Train Loss')
plt.plot(history_adam.history['val_loss'], label='Adam Validation Loss')
plt.title('Pérdida (Loss) durante el Entrenamiento')
plt.xlabel('Época')
plt.ylabel('Error Cuadrático Medio (MSE)')
plt.legend()
plt.grid(True)
plt.show()

# Visualización de predicciones
predicciones_sgd = model_sgd.predict(X_test)
predicciones_adam = model_adam.predict(X_test)

plt.figure(figsize=(12, 6))
plt.plot(Y_test, label='Valores Reales')
plt.plot(predicciones_sgd, label='Predicciones SGD', alpha=0.7)
plt.plot(predicciones_adam, label='Predicciones Adam', alpha=0.7)
plt.title('Predicciones de Serie Temporal en Datos de Prueba')
plt.xlabel('Punto en la Serie')
plt.ylabel('Valor')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# --- 1. Dataset: Serie Temporal Senoidal con Ruido ---
def generar_serie_sinusoidal(n_puntos, ventana):
    x = np.linspace(0, 20 * np.pi, n_puntos)
    y = np.sin(x) + 0.1 * np.random.randn(n_puntos)
    X, Y = [], []
    for i in range(len(y) - ventana):
        X.append(y[i:i+ventana])
        Y.append(y[i+ventana])
    return np.array(X), np.array(Y)

N_PUNTOS = 1000
VENTANA = 10
X_data, Y_data = generar_serie_sinusoidal(N_PUNTOS, VENTANA)

# Dividir en entrenamiento y prueba
split = int(N_PUNTOS * 0.8)
X_train, X_test = X_data[:split], X_data[split:]
Y_train, Y_test = Y_data[:split], Y_data[split:]

# --- 2. Modelo Base (Sin Atención) ---
modelo_base = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(VENTANA,)),
    tf.keras.layers.Dense(1)
])
modelo_base.compile(optimizer='adam', loss='mse')
print("Modelo Base:")
modelo_base.summary()

# --- 3. Modelo con Atención Local sobre Experiencias Pasadas ---

class CapaAtencionLocal(tf.keras.layers.Layer):
    def __init__(self, tam_buffer=5):
        super().__init__()
        self.tam_buffer = tam_buffer
        self.buffer = None
        self.buffer_errores = None

    def build(self, input_shape):
        self.buffer = self.add_weight(shape=(self.tam_buffer, input_shape[-1]),
                                      initializer='zeros',
                                      trainable=False,
                                      name='buffer_activaciones')
        self.buffer_errores = self.add_weight(shape=(self.tam_buffer,),
                                              initializer='ones',
                                              trainable=False,
                                              name='buffer_errores')

    def call(self, x, loss_actual):
        # Actualizar buffer
        activaciones = tf.squeeze(x, axis=0)
        self.buffer.assign(tf.concat([self.buffer[1:], [activaciones]], axis=0))
        self.buffer_errores.assign(tf.concat([self.buffer_errores[1:], [loss_actual]], axis=0))

        # Atender a entradas pasadas (menos error = más atención)
        pesos = 1.0 / (self.buffer_errores + 1e-6)
        pesos_normalizados = pesos / tf.reduce_sum(pesos)
        ponderado = tf.reduce_sum(self.buffer * tf.expand_dims(pesos_normalizados, -1), axis=0)

        # Concatenar entrada actual con atención del pasado
        salida = tf.concat([activaciones, ponderado], axis=0)
        return tf.expand_dims(salida, axis=0)

class ModeloAtencion(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.atencion = CapaAtencionLocal()
        self.densa1 = tf.keras.layers.Dense(32, activation='relu', input_shape=(VENTANA * 2,))
        self.salida = tf.keras.layers.Dense(1)

    def call(self, x, loss_actual):
        x_atendido = self.atencion(x, loss_actual)
        x1 = self.densa1(x_atendido)
        return self.salida(x1)

modelo_atencion = ModeloAtencion()
optimizer = tf.keras.optimizers.Adam()
loss_fn = tf.keras.losses.MeanSquaredError()

# --- 4. Entrenamiento Manual del Modelo con Atención ---
EPOCHS = 50
train_loss_hist = []

for epoch in range(EPOCHS):
    losses = []
    for xi, yi in zip(X_train, Y_train):
        xi = tf.expand_dims(xi, axis=0)
        yi = tf.expand_dims(yi, axis=0)

        with tf.GradientTape() as tape:
            pred = modelo_atencion(xi, tf.constant(1.0))  # Dummy loss inicial
            loss = loss_fn(yi, pred)

        gradients = tape.gradient(loss, modelo_atencion.trainable_variables)
        optimizer.apply_gradients(zip(gradients, modelo_atencion.trainable_variables))

        # Realimentar el loss real para la siguiente iteración
        _ = modelo_atencion(xi, loss)
        losses.append(loss.numpy())

    promedio = np.mean(losses)
    train_loss_hist.append(promedio)
    print(f"Época {epoch+1}/{EPOCHS} - Loss Atención: {promedio:.4f}")

# --- 5. Entrenamiento del Modelo Base ---
print("Entrenando modelo base...")
hist_base = modelo_base.fit(X_train, Y_train, epochs=EPOCHS, validation_data=(X_test, Y_test), verbose=0)

# --- 6. Evaluación ---
pred_base = modelo_base.predict(X_test)
loss_base = tf.keras.losses.mse(Y_test, pred_base).numpy().mean()

# Evaluar modelo con atención
preds_atencion = []
for xi in X_test:
    xi = tf.expand_dims(xi, axis=0)
    pred = modelo_atencion(xi, tf.constant(0.0))
    preds_atencion.append(pred.numpy().squeeze())

loss_atencion = tf.keras.losses.mse(Y_test, preds_atencion).numpy().mean()

# --- 7. Resultados ---
print("\n--- Comparación Final ---")
print(f"Loss Modelo Base: {loss_base:.4f}")
print(f"Loss Modelo con Atención Local: {loss_atencion:.4f}")

# --- 8. Visualización ---
plt.figure(figsize=(12, 6))
plt.plot(hist_base.history['loss'], label='Base - Train Loss')
plt.plot(train_loss_hist, label='Atención Local - Train Loss')
plt.title('Pérdida durante Entrenamiento')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.grid(True)
plt.show()

# Visualización de predicciones
plt.figure(figsize=(12, 6))
plt.plot(Y_test, label='Real')
plt.plot(pred_base, label='Predicción Base', alpha=0.7)
plt.plot(preds_atencion, label='Predicción Atención Local', alpha=0.7)
plt.title('Predicciones en Serie Temporal')
plt.xlabel('Punto')
plt.ylabel('Valor')
plt.legend()
plt.grid(True)
plt.show()